In [2]:
# ==========================================
# MODULE 4: MODELING & PREDICTION
# Linear Regression (Stock Returns Model)
# ==========================================

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ------------------------------------------
# 1️⃣ LOAD DATA
# ------------------------------------------

df = pd.read_csv("../data/processed/features_dataset.csv", low_memory=False)
df.columns = df.columns.str.strip()
df["Date"] = pd.to_datetime(df["Date"])
print([col for col in df.columns if "return" in col])
list(df)

['return_1d', 'return_5d', 'return_10d', 'return_21d', 'risk_adjusted_return']


/var/folders/kr/8p0vbgj13wvcz3hlcpdwqkh40000gn/T/ipykernel_42393/3758518858.py:17: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df["Date"] = pd.to_datetime(df["Date"])


['Ticker',
 'Date',
 'Close',
 'Open',
 'High',
 'Low',
 'return_1d',
 'return_5d',
 'return_10d',
 'return_21d',
 'ma_5',
 'ma_10',
 'ma_21',
 'price_ma5_ratio',
 'price_ma10_ratio',
 'price_ma21_ratio',
 'vol_5',
 'vol_10',
 'vol_21',
 'daily_range',
 'range_pct',
 'momentum_5',
 'momentum_10',
 'momentum_21',
 'momentum_5_pct',
 'momentum_10_pct',
 'momentum_21_pct',
 '52w_position',
 'gap',
 'risk_adjusted_return',
 'volume_avg_5',
 'volume_vol_5',
 'volume_momentum_5',
 'volume_momentum_5_pct',
 'volume_avg_10',
 'volume_vol_10',
 'volume_momentum_10',
 'volume_momentum_10_pct',
 'volume_avg_21',
 'volume_vol_21',
 'volume_momentum_21',
 'volume_momentum_21_pct',
 'volume_price_ratio',
 'ema_5',
 'ema_10',
 'ema_21',
 'ema_12',
 'ema_26',
 'macd',
 'macd_signal',
 'rsi_14',
 'close_ema5_ratio',
 'close_ema10_ratio',
 'close_ema21_ratio',
 'volume_ma5_ratio',
 'volume_ma10_ratio',
 'volume_ma21_ratio']

In [5]:
# ------------------------------------------
# 2️⃣ DEFINE TARGET (IMPORTANT)
# ------------------------------------------

target_col = "return_1d"

# ------------------------------------------
# 3️⃣ DEFINE FEATURES (IMPORTANT)
# ------------------------------------------

feature_cols = [
    "Close","Open","High","Low",
    "ma_5","ma_10","ma_21",
    "ema_5","ema_10","ema_21","ema_12","ema_26",
    "price_ma5_ratio","price_ma10_ratio","price_ma21_ratio",
    "momentum_5","momentum_10","momentum_21",
    "momentum_5_pct","momentum_10_pct","momentum_21_pct",
    "vol_5","vol_10","vol_21","range_pct",
    "volume_avg_5","volume_avg_10","volume_avg_21",
    "volume_momentum_5","volume_price_ratio",
    "rsi_14","macd","macd_signal",
    "risk_adjusted_return",
    "52w_position"
]

# ------------------------------------------
# 4️⃣ REMOVE MISSING TARGET ROWS
# ------------------------------------------

df = df.dropna(subset=[target_col])

# ------------------------------------------
# 5️⃣ TIME-BASED SPLIT
# ------------------------------------------

split_date = df["Date"].quantile(0.8)

train_df = df[df["Date"] <= split_date]
test_df  = df[df["Date"] > split_date]

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test  = test_df[feature_cols]
y_test  = test_df[target_col]

# ------------------------------------------
# 6️⃣ TRAIN MODEL
# ------------------------------------------

model = LinearRegression()
model.fit(X_train, y_train)

# ------------------------------------------
# 7️⃣ PREDICT
# ------------------------------------------

y_pred = model.predict(X_test)

# ------------------------------------------
# 8️⃣ EVALUATE MODEL
# ------------------------------------------

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("\n📊 MODEL PERFORMANCE")
print("RMSE:", rmse)
print("MAE :", mae)
print("R²  :", r2)

# ------------------------------------------
# 9️⃣ FEATURE IMPORTANCE
# ------------------------------------------

coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.coef_
}).sort_values(by="Coefficient", ascending=False)

print("\n📈 TOP POSITIVE DRIVERS:")
print(coef_df.head(10))

print("\n📉 TOP NEGATIVE DRIVERS:")
print(coef_df.tail(10))

# ------------------------------------------
# 🔟 BACKTEST + NEXT LEVEL METRICS
# ------------------------------------------

test_df = test_df.copy().reset_index(drop=True)

# Predictions
test_df["Predicted_Return"] = y_pred

# Signal (can tune threshold later)
test_df["Signal"] = (test_df["Predicted_Return"] > 0).astype(int)

# Strategy return
test_df["Strategy_Return"] = test_df["Signal"] * test_df[target_col]

# Cumulative returns
test_df["Cumulative_Market"] = (1 + test_df[target_col]).cumprod()
test_df["Cumulative_Strategy"] = (1 + test_df["Strategy_Return"]).cumprod()

# ------------------------------------------
# 🔥 ADVANCED METRICS
# ------------------------------------------

# PnL
test_df["PnL"] = test_df["Strategy_Return"]

# Drawdown
test_df["Rolling_Max"] = test_df["Cumulative_Strategy"].cummax()
test_df["Drawdown"] = (
    test_df["Cumulative_Strategy"] - test_df["Rolling_Max"]
) / test_df["Rolling_Max"]

# Volatility (21-day)
test_df["Volatility_21"] = test_df["Strategy_Return"].rolling(21).std()

# Rolling Sharpe
test_df["Sharpe_21"] = (
    test_df["Strategy_Return"].rolling(21).mean() /
    test_df["Strategy_Return"].rolling(21).std()
) * np.sqrt(252)

# Trade stats
test_df["Win"] = (test_df["Strategy_Return"] > 0).astype(int)
test_df["Loss"] = (test_df["Strategy_Return"] < 0).astype(int)
test_df["Trade"] = test_df["Signal"]

# ------------------------------------------
# 📊 SUMMARY METRICS
# ------------------------------------------

total_return = test_df["Cumulative_Strategy"].iloc[-1] - 1
market_return = test_df["Cumulative_Market"].iloc[-1] - 1

win_rate = test_df["Win"].sum() / test_df["Trade"].sum()

sharpe = (
    test_df["Strategy_Return"].mean() /
    test_df["Strategy_Return"].std()
) * np.sqrt(252)

max_drawdown = test_df["Drawdown"].min()

print("\n📊 ADVANCED STRATEGY METRICS")
print(f"Total Strategy Return: {total_return:.2%}")
print(f"Market Return: {market_return:.2%}")
print(f"Win Rate: {win_rate:.2%}")
print(f"Sharpe Ratio: {sharpe:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%}")

# ------------------------------------------
# SAVE FINAL OUTPUT
# ------------------------------------------

final_df = test_df[[
    "Ticker",
    "Date",
    "return_1d",
    "Predicted_Return",
    "Signal",
    "Strategy_Return",
    "Cumulative_Market",
    "Cumulative_Strategy",
    "Drawdown",
    "Sharpe_21",
    "Volatility_21"
]].copy()

final_df.to_csv("../data/processed/linear_model_predictions.csv", index=False)

print("\n✅ DONE: Model + strategy + metrics saved.")


📊 MODEL PERFORMANCE
RMSE: 0.013276287422186527
MAE : 0.00772230684327244
R²  : 0.6607564796340677

📈 TOP POSITIVE DRIVERS:
             Feature  Coefficient
10            ema_12     1.766739
31              macd     1.336264
11            ema_26     0.430475
12   price_ma5_ratio     0.067287
22            vol_10     0.058347
13  price_ma10_ratio     0.041925
18    momentum_5_pct     0.041139
19   momentum_10_pct     0.015252
0              Close     0.007760
21             vol_5     0.003695

📉 TOP NEGATIVE DRIVERS:
             Feature  Coefficient
3                Low    -0.000703
1               Open    -0.000925
14  price_ma21_ratio    -0.001367
20   momentum_21_pct    -0.001473
7              ema_5    -0.074931
23            vol_21    -0.079800
9             ema_21    -0.086332
24         range_pct    -0.118283
32       macd_signal    -0.636569
8             ema_10    -2.042031

📊 ADVANCED STRATEGY METRICS
Total Strategy Return: 972992756.56%
Market Return: -23.64%
Win Rate: 84.2

In [6]:
list(final_df)

['Ticker',
 'Date',
 'return_1d',
 'Predicted_Return',
 'Signal',
 'Strategy_Return',
 'Cumulative_Market',
 'Cumulative_Strategy',
 'Drawdown',
 'Sharpe_21',
 'Volatility_21']